# 📘 CIFAR-10 Image Classification Learning Project
## Build and Compare **ANN vs CNN** on CIFAR-10

This notebook is designed for **students and beginners** to learn:
- How image classification works
- Why **CNN performs better than ANN**
- How architecture impacts performance
- How training strategies improve results

🎯 **Learning Goal:** Understand the complete DL pipeline by **reading the markdown + running the ready code**.

# 🧠 Problem Statement
Build an image classification model on the **CIFAR-10 dataset** using:

1. **Artificial Neural Network (ANN)**
2. **Convolutional Neural Network (CNN)**

Then compare:
- Accuracy
- Loss curves
- Generalization
- Training strategies (dropout, batch norm, augmentation)

---
### 📦 CIFAR-10 Classes
Airplane, Automobile, Bird, Cat, Deer, Dog, Frog, Horse, Ship, Truck

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

print("TensorFlow version:", tf.__version__)

# 📥 Load Dataset
We use **CIFAR-10**, which contains **60,000 color images of size 32×32×3**.
- 50,000 training images
- 10,000 test images

In [ ]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()

class_names = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

print("Train shape:", x_train.shape)
print("Test shape:", x_test.shape)

## 🖼️ Visualize Sample Images

In [ ]:
plt.figure(figsize=(10,5))
for i in range(10):
    plt.subplot(2,5,i+1)
    plt.imshow(x_train[i])
    plt.title(class_names[y_train[i][0]])
    plt.axis("off")
plt.tight_layout()
plt.show()

# 🧹 Preprocessing
We normalize pixel values from **0–255 → 0–1** so training becomes stable.

In [ ]:
x_train_norm = x_train / 255.0
x_test_norm = x_test / 255.0

x_train_flat = x_train_norm.reshape(len(x_train_norm), -1)
x_test_flat = x_test_norm.reshape(len(x_test_norm), -1)

# 🔹 Part 1: ANN Model
ANN treats images as **flat vectors**, so it cannot preserve spatial features.
This helps students understand **why CNN is better for images**.

In [ ]:
ann_model = models.Sequential([
    layers.Dense(512, activation='relu', input_shape=(3072,)),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dense(10, activation='softmax')
])

ann_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

ann_history = ann_model.fit(
    x_train_flat, y_train,
    epochs=10,
    validation_split=0.1,
    batch_size=64
)

In [ ]:
ann_test_loss, ann_test_acc = ann_model.evaluate(x_test_flat, y_test)
print("ANN Test Accuracy:", ann_test_acc)

# 🔹 Part 2: CNN Model
CNN preserves **spatial relationships** using:
- Convolution layers
- Pooling
- Feature extraction
- Hierarchical learning

This is why CNN performs much better for image tasks.

In [ ]:
cnn_model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
])

cnn_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cnn_history = cnn_model.fit(
    x_train_norm, y_train,
    epochs=10,
    validation_split=0.1,
    batch_size=64
)

In [ ]:
cnn_test_loss, cnn_test_acc = cnn_model.evaluate(x_test_norm, y_test)
print("CNN Test Accuracy:", cnn_test_acc)

## 📈 Compare Learning Curves

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(ann_history.history['val_accuracy'], label='ANN Val Acc')
plt.plot(cnn_history.history['val_accuracy'], label='CNN Val Acc')
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("ANN vs CNN Validation Accuracy")
plt.legend()
plt.show()

# 🚀 Training Strategy Upgrade: Data Augmentation
This strategy improves generalization by generating transformed images.

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

aug_cnn_model = models.Sequential([
    data_augmentation,
    layers.Conv2D(32, 3, activation='relu', input_shape=(32,32,3)),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
])

aug_cnn_model.compile(optimizer='adam',
                      loss='sparse_categorical_crossentropy',
                      metrics=['accuracy'])

# Suggested optional run:
aug_history = aug_cnn_model.fit(x_train_norm, y_train, epochs=10, validation_split=0.1)

# 📊 Final Comparison Table

In [ ]:
comparison = pd.DataFrame({
    "Model": ["ANN", "CNN"],
    "Test Accuracy": [ann_test_acc, cnn_test_acc]
})
comparison

# 🎓 Student Learning Tasks
Try these tasks after understanding the notebook:

### ✅ Beginner Tasks
1. Increase ANN layers and observe performance
2. Change CNN filters from 32→64→128
3. Increase epochs to 20
4. Add **EarlyStopping**
5. Add **data augmentation training**

### Increase ANN Layers and Observe Performance

 `Dense` layer with 128 units and a `Dropout` layer to the Artificial Neural Network model. This increases the model's capacity, potentially allowing it to learn more complex patterns, but also increasing the risk of overfitting if not properly regularized. The model will also be trained for 20 epochs with `EarlyStopping`.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# Define EarlyStopping once for all tasks for consistency
early_stopping = callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Re-define ANN model with increased layers
ann_model_increased_layers = models.Sequential([
    layers.Dense(512, activation='relu', input_shape=(3072,)),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3), # Added new dropout layer
    layers.Dense(128, activation='relu'), # Added new dense layer
    layers.Dense(10, activation='softmax')
])

ann_model_increased_layers.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\nTraining ANN model with increased layers (epochs=20, EarlyStopping):")
ann_history_increased_layers = ann_model_increased_layers.fit(
    x_train_flat, y_train,
    epochs=20, # Increased epochs to 20
    validation_split=0.1,
    batch_size=64,
    callbacks=[early_stopping]
)

ann_test_loss_increased_layers, ann_test_acc_increased_layers = ann_model_increased_layers.evaluate(x_test_flat, y_test)
print(f"ANN Model (Increased Layers) Test Accuracy: {ann_test_acc_increased_layers:.4f}")

### Change CNN Filters from 32→64→128

 the Convolutional Neural Network model to use a progressively increasing number of filters in its `Conv2D` layers: 64, 128, and 256. Increasing the number of filters allows the CNN to learn more diverse features at each layer. Batch Normalization is also added after each convolutional layer to improve training stability and speed. The model will also be trained for 20 epochs with `EarlyStopping`.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# Ensure early_stopping is defined if running this cell independently
# early_stopping = callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Re-define CNN model with changed filters
cnn_model_changed_filters = models.Sequential([
    layers.Conv2D(64, (3,3), activation='relu', input_shape=(32,32,3)), # Changed filters from 32 to 64
    layers.BatchNormalization(), # Added BatchNormalization
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu'), # Changed filters from 64 to 128
    layers.BatchNormalization(), # Added BatchNormalization
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(256, (3,3), activation='relu'), # Changed filters from 128 to 256
    layers.BatchNormalization(), # Added BatchNormalization
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
])

cnn_model_changed_filters.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\nTraining CNN model with modified filters (epochs=20, EarlyStopping):")
cnn_history_changed_filters = cnn_model_changed_filters.fit(
    x_train_norm, y_train,
    epochs=20, # Increased epochs to 20
    validation_split=0.1,
    batch_size=64,
    callbacks=[early_stopping]
)

cnn_test_loss_changed_filters, cnn_test_acc_changed_filters = cnn_model_changed_filters.evaluate(x_test_norm, y_test)
print(f"CNN Model (Modified Filters) Test Accuracy: {cnn_test_acc_changed_filters:.4f}")

### Increase Epochs to 20 and Add EarlyStopping

These tasks involve adjusting the training parameters for both ANN and CNN models. By increasing the number of epochs to 20, the models have more opportunities to learn from the data. However, to prevent overfitting with more epochs, `EarlyStopping` is crucial. It monitors the validation loss and stops training if the loss doesn't improve for a specified number of epochs (patience=3), restoring the model weights from the best-performing epoch.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# Ensure early_stopping is defined
# early_stopping = callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Re-define ANN model (using increased layers for consistency with previous changes)
ann_model_epochs_es = models.Sequential([
    layers.Dense(512, activation='relu', input_shape=(3072,)),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

ann_model_epochs_es.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\nTraining ANN model with 20 epochs and EarlyStopping:")
ann_history_epochs_es = ann_model_epochs_es.fit(
    x_train_flat, y_train,
    epochs=20, # Increased epochs to 20
    validation_split=0.1,
    batch_size=64,
    callbacks=[early_stopping]
)

ann_test_loss_epochs_es, ann_test_acc_epochs_es = ann_model_epochs_es.evaluate(x_test_flat, y_test)
print(f"ANN Model (20 Epochs, EarlyStopping) Test Accuracy: {ann_test_acc_epochs_es:.4f}")

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# Ensure early_stopping is defined
# early_stopping = callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Re-define CNN model (using changed filters for consistency with previous changes)
cnn_model_epochs_es = models.Sequential([
    layers.Conv2D(64, (3,3), activation='relu', input_shape=(32,32,3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(256, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
])

cnn_model_epochs_es.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\nTraining CNN model with 20 epochs and EarlyStopping:")
cnn_history_epochs_es = cnn_model_epochs_es.fit(
    x_train_norm, y_train,
    epochs=20, # Increased epochs to 20
    validation_split=0.1,
    batch_size=64,
    callbacks=[early_stopping]
)

cnn_test_loss_epochs_es, cnn_test_acc_epochs_es = cnn_model_epochs_es.evaluate(x_test_norm, y_test)
print(f"CNN Model (20 Epochs, EarlyStopping) Test Accuracy: {cnn_test_acc_epochs_es:.4f}")

### Add Data Augmentation Training

Data augmentation is a powerful technique to improve model generalization by creating new training samples through random transformations of existing images. This helps the model become more robust to variations in input data. Here, I'll apply horizontal flips, rotations, and zooms to the training images for the CNN model. The augmented CNN will also use the updated filter sizes (64, 128) and be trained for 20 epochs with `EarlyStopping`.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# Ensure early_stopping is defined
# early_stopping = callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

aug_cnn_model = models.Sequential([
    data_augmentation,
    layers.Conv2D(64, 3, activation='relu', input_shape=(32,32,3)), # Consistent filter size
    layers.BatchNormalization(), # Added BatchNormalization
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, activation='relu'), # Consistent filter size
    layers.BatchNormalization(), # Added BatchNormalization
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
])

aug_cnn_model.compile(optimizer='adam',
                            loss='sparse_categorical_crossentropy',
                            metrics=['accuracy'])

print("\nTraining Augmented CNN model (epochs=20, EarlyStopping):")
aug_history = aug_cnn_model.fit(
    x_train_norm, y_train,
    epochs=20,
    validation_split=0.1,
    batch_size=64,
    callbacks=[early_stopping]
)

aug_cnn_test_loss, aug_cnn_test_acc = aug_cnn_model.evaluate(x_test_norm, y_test)
print(f"Augmented CNN Model Test Accuracy: {aug_cnn_test_acc:.4f}")

### Overall Model Comparison

Below is an updated comparison table including the results from the various modifications applied to the ANN and CNN models. The aim is to showcase the impact of each change on model performance.

In [ ]:
import pandas as pd

# Re-evaluate original models if their variables were overwritten or not available
# Fallback to 0 if variables are not defined (should not happen if cells are run sequentially)
original_ann_test_acc = globals().get('ann_test_acc', 0.0)
original_cnn_test_acc = globals().get('cnn_test_acc', 0.0)

comparison_detailed = pd.DataFrame({
    "Model Variation": [
        "Original ANN",
        "ANN (Increased Layers, 20 Epochs, ES)",
        "Original CNN",
        "CNN (64-128-256 Filters, BN, 20 Epochs, ES)",
        "ANN (20 Epochs, ES)",
        "CNN (20 Epochs, ES)",
        "Augmented CNN (64-128 Filters, BN, 20 Epochs, ES)"
    ],
    "Test Accuracy": [
        original_ann_test_acc,
        ann_test_acc_increased_layers,
        original_cnn_test_acc,
        cnn_test_acc_changed_filters,
        ann_test_acc_epochs_es,
        cnn_test_acc_epochs_es,
        aug_cnn_test_acc
    ]
})

print("\nDetailed Model Comparison:")
display(comparison_detailed.sort_values(by='Test Accuracy', ascending=False))

## Detailed Beginner Tasks Implementation

### Task 1: Increase ANN Layers and Observe Performance

I will add an additional `Dense` layer with 128 units and a `Dropout` layer to the Artificial Neural Network model. This increases the model's capacity, potentially allowing it to learn more complex patterns, but also increasing the risk of overfitting if not properly regularized. The model will also be trained for 20 epochs with `EarlyStopping`.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# Define EarlyStopping once for all tasks for consistency
early_stopping = callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Re-define ANN model with increased layers
ann_model_task1 = models.Sequential([
    layers.Dense(512, activation='relu', input_shape=(3072,)),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3), # Added new dropout layer
    layers.Dense(128, activation='relu'), # Added new dense layer
    layers.Dense(10, activation='softmax')
])

ann_model_task1.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\nTraining ANN model with increased layers (epochs=20, EarlyStopping):")
ann_history_task1 = ann_model_task1.fit(
    x_train_flat, y_train,
    epochs=20, # Increased epochs to 20
    validation_split=0.1,
    batch_size=64,
    callbacks=[early_stopping]
)

ann_test_loss_task1, ann_test_acc_task1 = ann_model_task1.evaluate(x_test_flat, y_test)
print(f"ANN Model (Increased Layers) Test Accuracy: {ann_test_acc_task1:.4f}")

### Task 2: Change CNN Filters from 32→64→128

I will modify the Convolutional Neural Network model to use a progressively increasing number of filters in its `Conv2D` layers: 64, 128, and 256. Increasing the number of filters allows the CNN to learn more diverse features at each layer. Batch Normalization is also added after each convolutional layer to improve training stability and speed. The model will also be trained for 20 epochs with `EarlyStopping`.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# Ensure early_stopping is defined if running this cell independently
# early_stopping = callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Re-define CNN model with changed filters
cnn_model_task2 = models.Sequential([
    layers.Conv2D(64, (3,3), activation='relu', input_shape=(32,32,3)), # Changed filters from 32 to 64
    layers.BatchNormalization(), # Added BatchNormalization
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu'), # Changed filters from 64 to 128
    layers.BatchNormalization(), # Added BatchNormalization
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(256, (3,3), activation='relu'), # Changed filters from 128 to 256
    layers.BatchNormalization(), # Added BatchNormalization
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
])

cnn_model_task2.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\nTraining CNN model with modified filters (epochs=20, EarlyStopping):")
cnn_history_task2 = cnn_model_task2.fit(
    x_train_norm, y_train,
    epochs=20, # Increased epochs to 20
    validation_split=0.1,
    batch_size=64,
    callbacks=[early_stopping]
)

cnn_test_loss_task2, cnn_test_acc_task2 = cnn_model_task2.evaluate(x_test_norm, y_test)
print(f"CNN Model (Modified Filters) Test Accuracy: {cnn_test_acc_task2:.4f}")

### Task 3 & 4: Increase Epochs to 20 and Add EarlyStopping

These tasks involve adjusting the training parameters for both ANN and CNN models. By increasing the number of epochs to 20, the models have more opportunities to learn from the data. However, to prevent overfitting with more epochs, `EarlyStopping` is crucial. It monitors the validation loss and stops training if the loss doesn't improve for a specified number of epochs (patience=3), restoring the model weights from the best-performing epoch.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# Ensure early_stopping is defined
# early_stopping = callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Re-define ANN model (same as Task 1 for comparison under these training conditions)
ann_model_task3_4 = models.Sequential([
    layers.Dense(512, activation='relu', input_shape=(3072,)),
    layers.Dropout(0.3),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')
])

ann_model_task3_4.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\nTraining ANN model with 20 epochs and EarlyStopping:")
ann_history_task3_4 = ann_model_task3_4.fit(
    x_train_flat, y_train,
    epochs=20, # Increased epochs to 20
    validation_split=0.1,
    batch_size=64,
    callbacks=[early_stopping]
)

ann_test_loss_task3_4, ann_test_acc_task3_4 = ann_model_task3_4.evaluate(x_test_flat, y_test)
print(f"ANN Model (20 Epochs, EarlyStopping) Test Accuracy: {ann_test_acc_task3_4:.4f}")

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# Ensure early_stopping is defined
# early_stopping = callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Re-define CNN model (same as Task 2 for comparison under these training conditions)
cnn_model_task3_4 = models.Sequential([
    layers.Conv2D(64, (3,3), activation='relu', input_shape=(32,32,3)),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(128, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(256, (3,3), activation='relu'),
    layers.BatchNormalization(),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
])

cnn_model_task3_4.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\nTraining CNN model with 20 epochs and EarlyStopping:")
cnn_history_task3_4 = cnn_model_task3_4.fit(
    x_train_norm, y_train,
    epochs=20, # Increased epochs to 20
    validation_split=0.1,
    batch_size=64,
    callbacks=[early_stopping]
)

cnn_test_loss_task3_4, cnn_test_acc_task3_4 = cnn_model_task3_4.evaluate(x_test_norm, y_test)
print(f"CNN Model (20 Epochs, EarlyStopping) Test Accuracy: {cnn_test_acc_task3_4:.4f}")

### Task 5: Add Data Augmentation Training

Data augmentation is a powerful technique to improve model generalization by creating new training samples through random transformations of existing images. This helps the model become more robust to variations in input data. Here, I'll apply horizontal flips, rotations, and zooms to the training images for the CNN model. The augmented CNN will also use the updated filter sizes (64, 128) and be trained for 20 epochs with `EarlyStopping`.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

# Ensure early_stopping is defined
# early_stopping = callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

aug_cnn_model_task5 = models.Sequential([
    data_augmentation,
    layers.Conv2D(64, 3, activation='relu', input_shape=(32,32,3)), # Consistent filter size
    layers.BatchNormalization(), # Added BatchNormalization
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, activation='relu'), # Consistent filter size
    layers.BatchNormalization(), # Added BatchNormalization
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.4),
    layers.Dense(10, activation='softmax')
])

aug_cnn_model_task5.compile(optimizer='adam',
                            loss='sparse_categorical_crossentropy',
                            metrics=['accuracy'])

print("\nTraining Augmented CNN model (epochs=20, EarlyStopping):")
aug_history_task5 = aug_cnn_model_task5.fit(
    x_train_norm, y_train,
    epochs=20,
    validation_split=0.1,
    batch_size=64,
    callbacks=[early_stopping]
)

aug_cnn_test_loss_task5, aug_cnn_test_acc_task5 = aug_cnn_model_task5.evaluate(x_test_norm, y_test)
print(f"Augmented CNN Model Test Accuracy: {aug_cnn_test_acc_task5:.4f}")

### Overall Task Summary and Comparison

Below is an updated comparison table including the results from the various modifications applied to the ANN and CNN models. Note that the original ANN and CNN models might not have been fully re-evaluated with all the changes in the previous cells, so these new evaluations provide fresh perspectives based on the specific task requirements.

In [ ]:
import pandas as pd

# Re-evaluate original models if their variables were overwritten or not available
# Fallback to 0 if variables are not defined (should not happen if cells are run sequentially)
original_ann_test_acc = globals().get('ann_test_acc', 0.0)
original_cnn_test_acc = globals().get('cnn_test_acc', 0.0)

comparison_detailed = pd.DataFrame({
    "Model Variation": [
        "Original ANN",
        "ANN (Increased Layers, 20 Epochs, ES)",
        "Original CNN",
        "CNN (64-128-256 Filters, BN, 20 Epochs, ES)",
        "ANN (20 Epochs, ES Only)",
        "CNN (20 Epochs, ES Only)",
        "Augmented CNN (64-128 Filters, BN, 20 Epochs, ES)"
    ],
    "Test Accuracy": [
        original_ann_test_acc,
        ann_test_acc_task1, # From Task 1
        original_cnn_test_acc,
        cnn_test_acc_task2, # From Task 2
        ann_test_acc_task3_4, # From Task 3 & 4 ANN
        cnn_test_acc_task3_4, # From Task 3 & 4 CNN
        aug_cnn_test_acc_task5 # From Task 5
    ]
})

print("\nDetailed Model Comparison:")
display(comparison_detailed.sort_values(by='Test Accuracy', ascending=False))

# ✅ Conclusion
- **ANN works**, but ignores image structure
- **CNN extracts spatial features**, so it performs significantly better
- **Training strategies** like dropout, batch norm, and augmentation improve results
- This project builds strong fundamentals for **computer vision interviews and deep learning projects**